# Signisa — ASL Citizen holistic extraction (CPU, internet ON)

Attach the ASL Citizen Kaggle mirror (**kaggle.com/datasets/abd0kamel/asl-citizen**).
Extraction is sharded (SHARD_INDEX / NUM_SHARDS, separate sessions) AND time-boxed:
the run stops cleanly after TIME_BUDGET_H hours and still SAVES — a SIGKILLed session
publishes nothing, so the budget is what makes long shards safe.

**Chaining a shard that didn't finish:** publish this version's output, attach it as an
input, and re-run with the same SHARD_INDEX. Prior versions' clips are COPIED into this
run's output before extracting the remainder, so **the newest version always carries the
shard's complete corpus to date** — chain on just the latest version, and attach only
each shard's latest version to `kaggle_prep.ipynb`.

The first cell is an integrity gate: a partial mirror hard-fails rather than silently
extracting a subset — fall back to the Microsoft download if it trips.

In [ ]:
# CONFIG — the only cell to edit
SHARD_INDEX = 0     # 0 .. NUM_SHARDS-1, one session each
NUM_SHARDS = 4
WORKERS = 4         # Kaggle CPU sessions have 4 cores
TIME_BUDGET_H = 10.5  # stop cleanly before the 12 h session cap so the version saves
MAX_SIDE = 640      # frame long side before MediaPipe; 0 = full res. ~1.25x faster but can
                    # lose marginal detections — A/B n_detected_frames on ~50 clips first

In [ ]:
!git clone -q https://github.com/dayan-battulga/signisa.git /kaggle/working/signisa-repo
%pip install -q "/kaggle/working/signisa-repo[live]"
!curl -sfL --create-dirs -o /kaggle/working/model/holistic_landmarker.task \
    https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task

In [ ]:
# Integrity gate: a partial mirror must hard-fail, never silently extract a subset
import shutil
from pathlib import Path

VIDEO_SUFFIXES = {".mp4", ".mov", ".mkv", ".webm", ".avi"}
MIN_VIDEOS = 80_000  # full ASL Citizen is ~83k clips

by_root = {}
for root in Path("/kaggle/input").iterdir():
    vids = [p for p in root.rglob("*") if p.suffix.lower() in VIDEO_SUFFIXES]
    if vids:
        by_root[root] = vids
assert by_root, "no video files under /kaggle/input — is the ASL Citizen mirror attached?"
DATASET_ROOT, videos = max(by_root.items(), key=lambda kv: len(kv[1]))

split_csvs = {p.stem: p for p in DATASET_ROOT.rglob("*.csv")
              if p.stem in ("train", "val", "test")}
print(f"{DATASET_ROOT}: {len(videos)} videos; split CSVs: "
      f"{ {k: str(v) for k, v in split_csvs.items()} }")
assert len(videos) >= MIN_VIDEOS, (
    f"only {len(videos)} videos (< {MIN_VIDEOS}) — partial mirror, fall back to the "
    "Microsoft ASL Citizen download instead of extracting a subset")
assert set(split_csvs) == {"train", "val", "test"}, (
    f"missing split CSVs (found {sorted(split_csvs)}) — partial mirror, use the "
    "Microsoft download")

# copy the split CSVs into the output so every shard is self-contained for prep
splits_out = Path("/kaggle/working/splits")
splits_out.mkdir(exist_ok=True)
for name, p in split_csvs.items():
    shutil.copy(p, splits_out / f"{name}.csv")

In [ ]:
# Fold prior versions of this shard into this run's output, then extract the rest.
# The newest version therefore always holds the COMPLETE corpus to date (round-2
# output ~4.5 GB — well under the caps), and chaining/prep only ever need it.
import csv, shutil

priors = sorted(Path("/kaggle/input").glob("*/shard_manifest.json"))
out = Path("/kaggle/working/extracted")
out.mkdir(exist_ok=True)
copied, prior_failures = 0, {}
for m in priors:
    for p in (m.parent / "extracted").glob("*.npz"):
        if not (out / p.name).exists():  # overlapping chain links carry identical files
            shutil.copy(p, out / p.name)
            copied += 1
    f = m.parent / "extracted" / "failures.csv"
    if f.exists():
        for row in csv.DictReader(f.open()):
            prior_failures[row["file"]] = row["reason"]
if prior_failures:  # merged log: these clips stay skipped and stay reported
    with (out / "failures.csv").open("w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["file", "reason"])
        w.writerows(sorted(prior_failures.items()))
if priors:
    print(f"folded in {len(priors)} prior version(s): {copied} npz copied, "
          f"{len(prior_failures)} known failures carried over")

# out-dir now holds everything done so far; the script's own resume skips it all
!python /kaggle/working/signisa-repo/scripts/extract_holistic.py \
    --videos-dir {DATASET_ROOT} \
    --out-dir /kaggle/working/extracted \
    --model /kaggle/working/model/holistic_landmarker.task \
    --workers {WORKERS} --shard-index {SHARD_INDEX} --num-shards {NUM_SHARDS} \
    --time-budget-h {TIME_BUDGET_H} --max-side {MAX_SIDE}

In [ ]:
# Totals + ONE merged manifest covering prior + new extractions
import json, sys

sys.path.insert(0, "/kaggle/working/signisa-repo/scripts")
from extract_holistic import pending_videos

npz = list(out.glob("*.npz"))
failures = out / "failures.csv"
n_failed = sum(1 for _ in csv.DictReader(failures.open())) if failures.exists() else 0
remaining = len(pending_videos(DATASET_ROOT, out, SHARD_INDEX, NUM_SHARDS))
size_gb = sum(p.stat().st_size for p in npz) / 1e9
manifest = {"shard_index": SHARD_INDEX, "num_shards": NUM_SHARDS,
            "n_extracted": len(npz), "n_new_this_run": len(npz) - copied,
            "n_failed": n_failed, "n_remaining": remaining,
            "n_shard_videos": len(videos[SHARD_INDEX::NUM_SHARDS]),
            "chained_on": [m.parent.name for m in priors], "size_gb": round(size_gb, 2)}
json.dump(manifest, open("/kaggle/working/shard_manifest.json", "w"), indent=1)
print(manifest)
assert size_gb < 20, "over the notebook-output budget — raise NUM_SHARDS"
if remaining:
    print(f"INCOMPLETE: {remaining} clips left — publish this version, attach JUST it "
          "as an input, and re-run with the same SHARD_INDEX to continue")